# Galactic Dynamics: N-Body Simulation
This notebook explores the morphological and kinematic dynamics of spiral galaxies using the Velocity Verlet integration algorithm. We will simulate the Winding Problem, implement Lin-Shu's Density Wave Theory, and test the kinematic effects of a Dark Matter Halo on the solar orbital period.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# --- CONSTANTS AND RESCALED UNITS ---
G = 6.67430e-11 # Gravitational constant (m^3 kg^-1 s^-2)
M_S = 1.989e30  # Solar mass (kg)
c = 3.086e19    # 1 kiloparsec in meters
# t_unit is the time it takes light to travel 1 parsec, adjusted so G=1
t_unit = np.sqrt(c**3 / (G * M_S)) 
v_unit = c / t_unit

# --- INITIAL GALAXY PARAMETERS ---
N_arm = 1500
theta = np.linspace(0, 2*np.pi, N_arm)
a = 1.5
b = 0.3
dispersion = 0.35
M_BH = 4.31*1e6 # Sagittarius A* mass in solar masses
x_bh = 0.0
y_bh = 0.0

# Generation of Arm 1 (Logarithmic Spiral)
r = a * np.exp(b * theta)
x = np.cos(theta) * r + np.random.normal(0, dispersion, N_arm)
y = np.sin(theta) * r + np.random.normal(0, dispersion, N_arm)
v_mag = np.sqrt(M_BH / r) # Keplerian velocity for circular orbits
vx = -np.sin(theta) * v_mag
vy =  np.cos(theta) * v_mag

# Generation of Arm 2 (Phase shifted by pi radians)
theta2 = np.linspace(0, 2*np.pi, N_arm) + np.pi
r2 = a * np.exp(b * theta)
x2 = np.cos(theta2) * r2 + np.random.normal(0, dispersion, N_arm)
y2 = np.sin(theta2) * r2 + np.random.normal(0, dispersion, N_arm)
v_mag2 = np.sqrt(M_BH / r2)
vx2 = -np.sin(theta2) * v_mag2
vy2 =  np.cos(theta2) * v_mag2

# Combine stars from both arms
x_total = np.concatenate((x, x2))
y_total = np.concatenate((y, y2))
vx_total = np.concatenate((vx, vx2))
vy_total = np.concatenate((vy, vy2))

In [ ]:
# --- SYSTEM ASSEMBLY (Sagittarius A* + Stars) ---

# 1. Mass Array: Sagittarius A* at index 0, 1 solar mass for the rest of the stars
star_masses = np.ones(2 * N_arm) 
masses = np.insert(star_masses, 0, M_BH)

# 2. Position Array: Insert the black hole at the origin (0,0)
x_complete = np.insert(x_total, 0, x_bh)
y_complete = np.insert(y_total, 0, y_bh)

# Group coordinates into an (N_total, 2) matrix
# Index 0 will always represent the Black Hole's coordinates
positions = np.column_stack((x_complete, y_complete)) 

# 3. Velocity Array: The black hole starts at rest
vx_complete = np.insert(vx_total, 0, 0.0)
vy_complete = np.insert(vy_total, 0, 0.0)
velocities = np.column_stack((vx_complete, vy_complete))

# Total number of bodies to simulate
N = len(masses)

## Phase 1: The Winding Problem (Differential Rotation)
In a Keplerian regime, the inner regions of a galaxy rotate faster than the outer regions. This simulation integrates the orbits to demonstrate how solid spiral arms suffer an irreversible kinematic shear, proving that galactic arms cannot be rigid material structures.

In [ ]:
# --- INTEGRATION: VELOCITY VERLET ALGORITHM ---

# Initialize accelerations
accel = np.zeros((N,2))
for i in range(1, N):
    r_vec = positions[i] - positions[0]
    r = np.linalg.norm(r_vec)
    # Gravity of Sagittarius A* on each star (G=1 due to rescaled units)
    accel[i] -= masses[0] * r_vec / r**3  

h = 0.000005
num_steps = 15000
trajectories = [] # List to store the position history for the animation
w = np.zeros((N,2)) # Predictor auxiliary variable for Verlet

for step in range(num_steps):
    # Save the current state of the system
    trajectories.append(positions.copy()) 

    # Step 1: New positions
    positions = positions + h * velocities + 0.5 * h**2 * accel
    
    # Step 2: Predictor velocity at half step
    w = velocities + 0.5 * h * accel
    
    # Step 3: Evaluate new accelerations at the new position
    accel = np.zeros((N,2))
    for i in range(1, N):
        r_vec = positions[i] - positions[0]
        r = np.linalg.norm(r_vec)
        accel[i] -= masses[0] * r_vec / r**3

    # Step 4: Final velocity correction
    velocities = w + 0.5 * h * accel

print(f"Simulation finished. Recorded steps: {len(trajectories)}")

In [ ]:
from IPython.display import HTML
import matplotlib as mpl

plt.style.use('dark_background') 

# 1. Figure configuration
fig, ax = plt.subplots(figsize=(6, 6)) 
ax.set_aspect('equal') 
ax.set_xlim(-13, 13) 
ax.set_ylim(-13, 13)
ax.set_title("The Winding Problem (Differential Rotation)")
ax.set_xlabel("Distance (kpc)")
ax.set_ylabel("Distance (kpc)")

initial_pos = trajectories[0]

# 2. Graphics objects initialization
black_hole = ax.scatter(initial_pos[0,0], initial_pos[0,1], color='white', s=100, label='Sagittarius A*', zorder=5) 
stars = ax.scatter(initial_pos[1:,0], initial_pos[1:,1], color='yellow', s=3, label='Stars', zorder=2) 

plt.legend(loc="upper right")

# 3. Update function for each frame
def update(i):
    # Skip iterations to prevent animation saturation
    current_pos = trajectories[i*50] 
    black_hole.set_offsets(current_pos[0:1]) 
    stars.set_offsets(current_pos[1:]) 
    return black_hole, stars

# 4. Animation creation
ani = animation.FuncAnimation(fig, update, frames=len(trajectories)//50, interval=20, blit=True) 

# Required configuration to render properly in Jupyter Notebook
plt.close() 
mpl.rcParams['animation.embed_limit'] = 100 
display(HTML(ani.to_jshtml()))

# --- SAVING SPECIFIC FRAMES FOR DOCUMENTATION ---
save_steps = [0, len(trajectories)//3, 2*len(trajectories)//3, -1]
file_names = ['winding_1_start.png', 'winding_2_mid1.png', 'winding_3_mid2.png', 'winding_4_final.png']

for step, name in zip(save_steps, file_names):
    frame_pos = trajectories[step]
    
    fig_f, ax_f = plt.subplots(figsize=(5, 5))
    ax_f.set_aspect('equal')
    ax_f.set_xlim(-13, 13)
    ax_f.set_ylim(-13, 13)
    fig_f.patch.set_facecolor('black')
    ax_f.set_facecolor('black')
    
    ax_f.scatter(frame_pos[0, 0], frame_pos[0, 1], color='white', s=100, zorder=5)
    ax_f.scatter(frame_pos[1:, 0], frame_pos[1:, 1], color='yellow', s=2, alpha=0.8, zorder=2)
    
    ax_f.set_title(f"Iteration: {step if step != -1 else len(trajectories)}", color='white', fontsize=10)
    ax_f.set_xlabel("x (kpc)", fontsize=8)
    ax_f.set_ylabel("y (kpc)", fontsize=8)
    ax_f.grid(alpha=0.2)
    
    plt.tight_layout()
    plt.savefig(name, dpi=300, bbox_inches='tight', facecolor=fig_f.get_facecolor())
    plt.close()

## Phase 2: Lin-Shu Density Wave Theory
To solve the structural collapse, we implement Lin-Shu's kinematic model. By placing stars in synchronized elliptical orbits with a progressive periapsis phase shift, we create a stable "cosmic traffic jam". We also introduce a test body (the Sun) at 8 kpc to evaluate the luminous mass model.

In [ ]:
# --- KINEMATIC MODEL: LIN-SHU DENSITY WAVE THEORY ---

N_arm = 1500
eccen = 0.3
a_min = 1.5
a_max = 10
# Modulation factor: determines the opening degree of the spiral pattern
k = 0.5 

# Random semi-major axis for each star
a = np.random.uniform(a_min, a_max, N_arm) 
# Linear phase shift of the periapsis to create the "cosmic traffic jam"
theta_0 = k * a 
theta = np.random.uniform(0, 2*np.pi, N_arm)

# Polar equation of the ellipse
p = a * (1 - eccen**2)
r = p / (1 + eccen * np.cos(theta - theta_0)) 
x = np.cos(theta) * r
y = np.sin(theta) * r

# Calculation of radial and tangential components of orbital velocity
v_r1 = np.sqrt(M_BH / p) * eccen * np.sin(theta - theta_0) 
v_t1 = np.sqrt(M_BH / p) * (1 + eccen * np.cos(theta - theta_0)) 
# Transformation to Cartesian coordinates
vx = v_r1 * np.cos(theta) - v_t1 * np.sin(theta)
vy = v_r1 * np.sin(theta) + v_t1 * np.cos(theta)

# Same process for the second arm (phase shifted by pi radians)
theta_0_arm2 = theta_0 + np.pi
theta2 = np.random.uniform(0, 2*np.pi, N_arm)
r2 = p / (1 + eccen * np.cos(theta2 - theta_0_arm2)) 
x2 = np.cos(theta2) * r2
y2 = np.sin(theta2) * r2

v_r2 = np.sqrt(M_BH / p) * eccen * np.sin(theta2 - theta_0_arm2) 
v_t2 = np.sqrt(M_BH / p) * (1 + eccen * np.cos(theta2 - theta_0_arm2)) 
vx2 = v_r2 * np.cos(theta2) - v_t2 * np.sin(theta2)
vy2 = v_r2 * np.sin(theta2) + v_t2 * np.cos(theta2)

# --- INCLUSION OF THE TEST BODY (SUN) ---
# Placed at 8 kpc with a purely circular orbit to measure the orbital period
x_sun = 8.0
y_sun = 0.0
v_sun = np.sqrt(M_BH / x_sun)
vx_sun = 0.0
vy_sun = v_sun

# System assembly (Index 0: Black Hole, Index 1: Sun, Index 2+: Stars)
x_total = np.concatenate(([x_sun], x, x2))
y_total = np.concatenate(([y_sun], y, y2))
x_complete = np.insert(x_total, 0, x_bh)
y_complete = np.insert(y_total, 0, y_bh)

star_masses = np.ones(1 + 2 * N_arm) 
masses = np.insert(star_masses, 0, M_BH)

N = len(masses)
positions = np.column_stack((x_complete, y_complete))

vx_total = np.concatenate(([vx_sun], vx, vx2))
vy_total = np.concatenate(([vy_sun], vy, vy2))
vx_complete = np.insert(vx_total, 0, 0.0) 
vy_complete = np.insert(vy_total, 0, 0.0)
velocities = np.column_stack((vx_complete, vy_complete))

In [ ]:
# --- INTEGRATION AND SOLAR PERIOD MEASUREMENT (LUMINOUS MASS ONLY) ---

accel = np.zeros((N,2))
for i in range(1, N):
    r_vec = positions[i] - positions[0]
    r = np.linalg.norm(r_vec)
    accel[i] -= masses[0] * r_vec / (r**2)**1.5  

h = 0.0000025
num_steps = 33000
trajectories = [] 
w = np.zeros((N,2)) 

# Variables to record the time step when the Sun completes one orbit
sun_orbit_step = 0
prev_y_sun = positions[1, 1] 
orbit_completed = False

for step in range(num_steps):
    trajectories.append(positions.copy()) 

    positions = positions + h * velocities + 0.5 * h**2 * accel
    w = velocities + 0.5 * h * accel
    
    accel = np.zeros((N,2))
    for i in range(1, N):
        r_vec = positions[i] - positions[0] 
        r = np.linalg.norm(r_vec)
        accel[i] -= masses[0] * r_vec / (r**2)**1.5

    velocities = w + 0.5 * h * accel

    # Geometric check: The Sun crosses the positive y half-plane (y >= 0)
    current_y_sun = positions[1, 1]
    current_x_sun = positions[1, 0]

    if not orbit_completed and prev_y_sun < 0 and current_y_sun >= 0 and current_x_sun > 0:
        sun_orbit_step = step
        orbit_completed = True
        
    prev_y_sun = current_y_sun

# --- QUANTITATIVE RESULTS ---
if orbit_completed:
    simulated_time = sun_orbit_step * h
    t_unit_years = t_unit / (3600 * 24 * 365.25) # Conversion from seconds to years
    period_years = (simulated_time * t_unit_years) / 1e6 # Scale in millions of years
    
    real_period = 230.0 # Empirical estimation in the Milky Way
    relative_error = abs(period_years - real_period) / real_period * 100
    
    print(f"Log: The Sun completed an orbit at step {sun_orbit_step}.")
    print(f"Simulated orbital period (Luminous Model): {period_years:.2f} million years.")
    print(f"Expected empirical period: ~{real_period} million years.")
    print(f"Relative error: {relative_error:.2f}% (Evidencing the missing mass)")
else:
    print("The Sun did not complete an orbit. Increase num_steps.")

In [ ]:
# --- ANIMATION AND TRACKING: LIN-SHU MODEL ---
from IPython.display import HTML
import matplotlib as mpl

plt.style.use('dark_background')

# 1. Animated figure configuration
fig, ax = plt.subplots(figsize=(6, 6)) 
ax.set_aspect('equal') 
ax.set_xlim(-13, 13) 
ax.set_ylim(-13, 13)
ax.set_title("Lin-Shu Density Waves")
ax.set_xlabel("Distance (kpc)")
ax.set_ylabel("Distance (kpc)")

initial_pos = trajectories[0]

# Create the color array based on the semi-major axis for both arms
a_total = np.concatenate((a, a))

# 2. Graphics objects initialization
black_hole = ax.scatter(initial_pos[0,0], initial_pos[0,1], color='white', s=100, label='Sagittarius A*', zorder=5)
blue_star = ax.scatter(initial_pos[1,0], initial_pos[1,1], color='cyan', s=30, zorder=6, label='Sun')

# Continuous color mapping (cmap) to highlight the wave structure
stars = ax.scatter(initial_pos[2:,0], initial_pos[2:,1], 
                       c=a_total, cmap='RdYlBu', s=3, alpha=0.8, zorder=2)

# Orbital track of the Sun
track_x, track_y = [], []
track_line, = ax.plot([], [], color='cyan', alpha=0.4, linewidth=1.5, zorder=3)

plt.legend(loc="upper right")

# 3. Update function for the animation
def update(i):
    if i == 0:
        track_x.clear()
        track_y.clear()
        
    current_pos = trajectories[i*50] 
    black_hole.set_offsets(current_pos[0:1]) 
    blue_star.set_offsets(current_pos[1:2]) 
    stars.set_offsets(current_pos[2:]) 

    # Track update
    track_x.append(current_pos[1, 0])
    track_y.append(current_pos[1, 1])
    track_line.set_data(track_x, track_y)

    return black_hole, blue_star, stars, track_line

ani = animation.FuncAnimation(fig, update, frames=len(trajectories)//50, interval=20, blit=True) 

# Render configuration for Jupyter
plt.close() 
mpl.rcParams['animation.embed_limit'] = 300 

print("Generating Lin-Shu GIF (this may take a few seconds)...")
ani.save('lin_shu_galaxy.gif', writer='pillow', fps=30)
print("GIF successfully saved!")

display(HTML(ani.to_jshtml()))

# --- SAVING STATIC FRAMES FOR DOCUMENTATION ---
save_steps_ls = [0, len(trajectories)-1]
file_names_ls = ['lin_shu_1_start.png', 'lin_shu_2_steady_state.png']

# Reconstruct the full Sun track for the final frames
full_track_x = [pos[1, 0] for pos in trajectories]
full_track_y = [pos[1, 1] for pos in trajectories]

for step, name in zip(save_steps_ls, file_names_ls):
    frame_pos = trajectories[step]
    
    fig_ls, ax_ls = plt.subplots(figsize=(6, 6))
    ax_ls.set_aspect('equal')
    ax_ls.set_xlim(-13, 13)
    ax_ls.set_ylim(-13, 13)
    fig_ls.patch.set_facecolor('black')
    ax_ls.set_facecolor('black')
    
    ax_ls.scatter(frame_pos[0,0], frame_pos[0,1], color='white', s=100, zorder=5)
    ax_ls.plot(full_track_x[:step+1], full_track_y[:step+1], color='cyan', alpha=0.5, linewidth=1.5, zorder=3)
    ax_ls.scatter(frame_pos[1,0], frame_pos[1,1], color='cyan', s=40, zorder=6)
    ax_ls.scatter(frame_pos[2:,0], frame_pos[2:,1], c=a_total, cmap='RdYlBu', s=3, alpha=0.8, zorder=2)
    
    time_text = "t = 0" if step == 0 else "Steady State"
    ax_ls.set_title(f"Density Waves ({time_text})", color='white', fontsize=12)
    ax_ls.set_xlabel("x (kpc)", fontsize=10)
    ax_ls.set_ylabel("y (kpc)", fontsize=10)
    ax_ls.grid(alpha=0.15)
    
    plt.tight_layout()
    plt.savefig(name, dpi=300, bbox_inches='tight', facecolor=fig_ls.get_facecolor())
    plt.close()

print("Static Lin-Shu images successfully saved!")

## Phase 3: Dark Matter Halo & Kinematic Correction
The purely luminous model significantly overestimates the Sun's orbital period. To rectify this and match the empirical ~230 million year period, we introduce an extended Dark Matter halo. We apply a bifurcated potential to prevent the apsidal precession from destroying the Lin-Shu geometric model.

In [ ]:
# --- DARK MATTER HALO IMPLEMENTATION ---

# Extended mass model parameters
R_c = 2.5     # Core radius in kpc
v_0_sq = 1.1e10 # Velocity factor squared

def effective_mass(radius):
    # Central black hole mass + mass enclosed by the dark halo at distance r
    return M_BH + v_0_sq * (radius**3) / (R_c**2 + radius**2)

N_arm_dm = 750
eccen_dm = 0.3
a_min_dm = 1.5
a_max_dm = 10
k_dm = 0.5 

a_dm = np.random.uniform(a_min_dm, a_max_dm, N_arm_dm)
theta_0_dm = k_dm * a_dm 
theta_dm = np.random.uniform(0, 2*np.pi, N_arm_dm)

p_dm = a_dm * (1 - eccen_dm**2)
r_dm = p_dm / (1 + eccen_dm * np.cos(theta_dm - theta_0_dm))
x_dm = np.cos(theta_dm) * r_dm
y_dm = np.sin(theta_dm) * r_dm

# THE STARS: Their initial velocities are calculated assuming ONLY the Black Hole.
# This is required to prevent destroying the closed orbits of the Lin-Shu model.
v_r1_dm = np.sqrt(M_BH / p_dm) * eccen_dm * np.sin(theta_dm - theta_0_dm)
v_t1_dm = np.sqrt(M_BH / p_dm) * (1 + eccen_dm * np.cos(theta_dm - theta_0_dm))
vx_dm = v_r1_dm * np.cos(theta_dm) - v_t1_dm * np.sin(theta_dm)
vy_dm = v_r1_dm * np.sin(theta_dm) + v_t1_dm * np.cos(theta_dm)

# Arm 2
theta_0_arm2_dm = theta_0_dm + np.pi
theta2_dm = np.random.uniform(0, 2*np.pi, N_arm_dm)
r2_dm = p_dm / (1 + eccen_dm * np.cos(theta2_dm - theta_0_arm2_dm))
x2_dm = np.cos(theta2_dm) * r2_dm
y2_dm = np.sin(theta2_dm) * r2_dm

v_r2_dm = np.sqrt(M_BH / p_dm) * eccen_dm * np.sin(theta2_dm - theta_0_arm2_dm)
v_t2_dm = np.sqrt(M_BH / p_dm) * (1 + eccen_dm * np.cos(theta2_dm - theta_0_arm2_dm))
vx2_dm = v_r2_dm * np.cos(theta2_dm) - v_t2_dm * np.sin(theta2_dm)
vy2_dm = v_r2_dm * np.sin(theta2_dm) + v_t2_dm * np.cos(theta2_dm)

# THE SUN: This test body DOES experience the effective mass of the Dark Halo
x_sun_dm = 8.0
y_sun_dm = 0.0
v_sun_dm = np.sqrt(effective_mass(x_sun_dm) / x_sun_dm) 
vx_sun_dm = 0.0
vy_sun_dm = v_sun_dm

# Final matrix assembly
x_total_dm = np.concatenate(([x_sun_dm], x_dm, x2_dm))
y_total_dm = np.concatenate(([y_sun_dm], y_dm, y2_dm))
x_complete_dm = np.insert(x_total_dm, 0, x_bh)
y_complete_dm = np.insert(y_total_dm, 0, y_bh)

star_masses_dm = np.ones(1 + 2 * N_arm_dm) 
masses_dm = np.insert(star_masses_dm, 0, M_BH)

N_dm = len(masses_dm)
positions_dm = np.column_stack((x_complete_dm, y_complete_dm))

vx_total_dm = np.concatenate(([vx_sun_dm], vx_dm, vx2_dm))
vy_total_dm = np.concatenate(([vy_sun_dm], vy_dm, vy2_dm))
vx_complete_dm = np.insert(vx_total_dm, 0, 0.0) 
vy_complete_dm = np.insert(vy_total_dm, 0, 0.0)
velocities_dm = np.column_stack((vx_complete_dm, vy_complete_dm))

# Color array for rendering based on the semi-major axis
a_color_dm = np.concatenate((a_dm, a_dm))

In [ ]:
# --- INTEGRATION WITH POTENTIAL BIFURCATION ---

# Drastically reduce time step due to higher orbital velocities
h_dm = 0.00000005 
num_steps_dm = 75000 
step_jump_dm = 200 # Store state every 200 iterations to save RAM
trajectories_dm = [] 
w_dm = np.zeros((N_dm,2)) 

sun_orbit_step_dm = 0
prev_y_sun_dm = positions_dm[1, 1] 
orbit_completed_dm = False

def calc_bifurcated_accel(pos):
    """
    Evaluates gravitational forces applying a bifurcation:
    - The Sun (index 1) feels the full potential (Black Hole + Halo).
    - Arm stars (index > 1) only feel the point keplerian potential.
    This prevents apsidal precession and the destruction of the density wave model.
    """
    accel = np.zeros((N_dm,2))
    for i in range(1, N_dm):
        r_vec = pos[i] - pos[0]
        r = np.linalg.norm(r_vec)
        
        a_black_hole = -masses_dm[0] * r_vec / r**3
        
        if i == 1: 
            a_halo = -v_0_sq * r_vec / (R_c**2 + r**2)
            accel[i] = a_black_hole + a_halo
        else:      
            accel[i] = a_black_hole
            
    return accel

accel_dm = calc_bifurcated_accel(positions_dm)

for step in range(num_steps_dm):
    if step % step_jump_dm == 0: 
        trajectories_dm.append(positions_dm.copy()) 
    
    positions_dm = positions_dm + h_dm * velocities_dm + 0.5 * h_dm**2 * accel_dm
    w_dm = velocities_dm + 0.5 * h_dm * accel_dm
    
    accel_dm = calc_bifurcated_accel(positions_dm)
    velocities_dm = w_dm + 0.5 * h_dm * accel_dm
    
    current_y_sun = positions_dm[1, 1]
    current_x_sun = positions_dm[1, 0]
    
    if not orbit_completed_dm and prev_y_sun_dm < 0 and current_y_sun >= 0 and current_x_sun > 0:
        sun_orbit_step_dm = step
        orbit_completed_dm = True
        
    prev_y_sun_dm = current_y_sun

print(f"Integration finished. Saved frames: {len(trajectories_dm)}")

if orbit_completed_dm:
    simulated_time = sun_orbit_step_dm * h_dm
    t_unit_years = t_unit / (3600 * 24 * 365.25)
    period_years = (simulated_time * t_unit_years) / 1e6 
    real_period = 230.0 
    relative_error = abs(period_years - real_period) / real_period * 100
    
    print(f"\n--- RESULTS: DARK MATTER HALO MODEL ---")
    print(f"Rectified orbital period: {period_years:.2f} million years.")
    print(f"Real empirical period: ~{real_period} million years.")
    print(f"Relative error: {relative_error:.2f}%")

In [ ]:
# --- ANIMATION OF THE SYSTEM WITH DARK MATTER HALO ---
from IPython.display import HTML
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.animation as animation

plt.close('all') 
plt.style.use('dark_background') 

# 1. Figure configuration
fig, ax = plt.subplots(figsize=(7, 7)) 
ax.set_aspect('equal') 
ax.set_xlim(-13, 13) 
ax.set_ylim(-13, 13)
ax.set_title("Bifurcated Dynamics: Density Waves + Sun with Dark Halo")
ax.set_xlabel("Distance (kpc)")
ax.set_ylabel("Distance (kpc)")

initial_pos_dm = trajectories_dm[0]

# 2. Graphics objects initialization
black_hole_dm = ax.scatter(initial_pos_dm[0,0], initial_pos_dm[0,1], color='white', s=100, label='Sagittarius A*', zorder=5)
sun_star_dm = ax.scatter(initial_pos_dm[1,0], initial_pos_dm[1,1], color='cyan', s=40, label='Sun (8 kpc)', zorder=6)

# Render the spiral arms using the color array a_color_dm
real_stars_dm = ax.scatter(initial_pos_dm[2:,0], initial_pos_dm[2:,1], 
                              c=a_color_dm, cmap='RdYlBu', s=4, alpha=0.9, zorder=2, label='Stars (Spiral)')

# Orbital track of the Sun (only showing the last 200 points to prevent saturation)
track_x_dm, track_y_dm = [], []
track_line_dm, = ax.plot([], [], color='cyan', alpha=0.4, linewidth=1.5, zorder=3)

plt.legend(loc="upper right")

# 3. Update function for the animation
def update_dm(i):
    if i == 0:
        track_x_dm.clear() 
        track_y_dm.clear() 
        
    current_pos = trajectories_dm[i] 
    
    black_hole_dm.set_offsets(current_pos[0:1]) 
    sun_star_dm.set_offsets(current_pos[1:2]) 
    real_stars_dm.set_offsets(current_pos[2:]) 
    
    # Update track and limit to the last 200 frames for a "trail" effect
    track_x_dm.append(current_pos[1, 0])
    track_y_dm.append(current_pos[1, 1])
    track_line_dm.set_data(track_x_dm[-200:], track_y_dm[-200:])
    
    return black_hole_dm, sun_star_dm, real_stars_dm, track_line_dm

# 4. Animation creation and rendering
ani_dm = animation.FuncAnimation(fig, update_dm, frames=len(trajectories_dm), interval=30, blit=False) 

mpl.rcParams['animation.embed_limit'] = 100 

print("Generating Dark Matter galaxy GIF (this may take a minute)...")
ani_dm.save('dark_matter_galaxy.gif', writer='pillow', fps=30)
print("GIF successfully saved in the working directory!")


plt.close()

display(HTML(ani_dm.to_jshtml()))

In [ ]:
# --- KINEMATIC ANALYSIS AND ROTATION CURVES ---
import numpy as np
import matplotlib.pyplot as plt

# 1. Generation of continuous theoretical curves
r_theoretical = np.linspace(0.05, 12, 500)

# Keplerian Theory (Classical decay)
v_kepler_sim = np.sqrt(M_BH / r_theoretical)
v_kepler_km_s = v_kepler_sim * (v_unit / 1000)

# Dark Halo Theory (Asymptotic flattening)
theoretical_effective_mass = M_BH + v_0_sq * (r_theoretical**3) / (R_c**2 + r_theoretical**2)
v_dm_sim_theo = np.sqrt(theoretical_effective_mass / r_theoretical)
v_dm_km_s = v_dm_sim_theo * (v_unit / 1000)

# 2. Data extraction from the last frame of the simulation
# Test body (Sun)
v_sun_sim = np.sqrt(velocities_dm[1, 0]**2 + velocities_dm[1, 1]**2) 
v_sun_km_s = v_sun_sim * (v_unit / 1000)

# Galactic arm stars
star_radii = np.sqrt(positions_dm[2:, 0]**2 + positions_dm[2:, 1]**2)
star_vel_sim = np.sqrt(velocities_dm[2:, 0]**2 + velocities_dm[2:, 1]**2) 
star_vel_km_s = star_vel_sim * (v_unit / 1000)

# 3. Comparative Graphical Representation
plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(star_radii, star_vel_km_s, color='white', s=5, alpha=0.3, 
           label='Simulated Stars (Keplerian Regime)', zorder=2)

ax.scatter([8.0], [v_sun_km_s], color='yellow', s=200, marker='*', edgecolor='orange',
           label='Simulated Sun (Influenced by D.M.)', zorder=5)

ax.plot(r_theoretical, v_kepler_km_s, color='red', linestyle='--', linewidth=2.5, 
        label='Theoretical Model: Point Luminous Mass')
ax.plot(r_theoretical, v_dm_km_s, color='cyan', linewidth=2.5, 
        label='Theoretical Model: Dark Matter Halo')

ax.set_title("Kinematic Analysis: Orbital Velocity Discrepancy", fontsize=14, pad=15)
ax.set_xlabel("Distance to Galactic Center (kpc)", fontsize=12)
ax.set_ylabel("Orbital Velocity (km/s)", fontsize=12)
ax.legend(fontsize=14, loc='center right')
ax.grid(alpha=0.15)

plt.tight_layout()
plt.savefig('rotation_curve.png', dpi=300) 
plt.show()

## Limitation: Apsidal Precession Collapse
Finally, we demonstrate the inherent limitation of applying an extended non-Keplerian potential (Dark Matter) to the entire kinematic density wave model. Without the bifurcation, the orbits precess, blurring the density wave and collapsing the macroscopic structure.

In [ ]:
# --- LIMITATION EXPERIMENT: COLLAPSE BY APSIDAL PRECESSION ---
# We demonstrate what happens if we apply the extended mass of Dark Matter
# to ALL stars comprising the Lin-Shu geometric spiral.

import copy

# Copy the exact initial conditions from the Dark Matter scenario
broken_pos = copy.deepcopy(trajectories_dm[0])
broken_vel = copy.deepcopy(velocities_dm)
broken_accel = np.zeros((N_dm, 2))

def calc_real_accel(pos):
    """Here we eliminate the bifurcation: the whole galaxy feels Dark Matter"""
    accel = np.zeros((N_dm,2))
    for i in range(1, N_dm):
        r_vec = pos[i] - pos[0]
        r = np.linalg.norm(r_vec)
        a_black_hole = -masses_dm[0] * r_vec / r**3
        
        # The halo affects all baryonic matter without exception
        a_halo = -v_0_sq * r_vec / (R_c**2 + r**2)
        accel[i] = a_black_hole + a_halo
    return accel

# Integrate a reduced number of iterations (enough to observe the collapse)
h_broken = 0.00000005
broken_accel = calc_real_accel(broken_pos)
w_broken = np.zeros((N_dm, 2))

for step in range(15000):
    broken_pos = broken_pos + h_broken * broken_vel + 0.5 * h_broken**2 * broken_accel
    w_broken = broken_vel + 0.5 * h_broken * broken_accel
    broken_accel = calc_real_accel(broken_pos)
    broken_vel = w_broken + 0.5 * h_broken * broken_accel

# Rendering the final disjointed state
fig, ax = plt.subplots(figsize=(5, 5))
ax.set_aspect('equal')
ax.set_xlim(-13, 13)
ax.set_ylim(-13, 13)
fig.patch.set_facecolor('black')
ax.set_facecolor('black')

ax.scatter(broken_pos[0,0], broken_pos[0,1], color='white', s=80, zorder=5)
ax.scatter(broken_pos[2:,0], broken_pos[2:,1], color='yellow', s=3, alpha=0.6, zorder=2)

ax.set_title("Lin-Shu Model Collapse\n(Complete Galaxy Subjected to D.M.)", color='white', fontsize=10)
ax.set_xlabel("x (kpc)", color='white')
ax.set_ylabel("y (kpc)", color='white')
ax.tick_params(colors='white')
ax.grid(alpha=0.15)

plt.tight_layout()
plt.savefig('lin_shu_collapsed.png', dpi=300, bbox_inches='tight', facecolor='black')
plt.close()